In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import util.functions as functions
from models.FNO import CAPE_FNO
import pybamm
import flax

In [2]:
from util.FNO_util import preprocess_data, train_test_split, remove_padding, normalise_diffusion
from util.postprocess import filter_anode_cathode, calc_error_metrics, calc_error_metrics_all

In [3]:
family = "CC"
N_total = 33000
parameter_name = "Chen2020"
data = np.load(f"../data/{parameter_name}_{family}_{N_total}.npz")
random_seed = 42
test_ratio = 0.1

train_data, test_data = train_test_split(data, N_total=N_total, test_ratio=test_ratio, seed=random_seed)

In [4]:
train_I = np.array(train_data["current"])
test_I = np.array(test_data["current"])

### Anode data ###
train_cn_anode = np.array(train_data["cn_anode"])
test_cn_anode = np.array(test_data["cn_anode"])
train_c0_anode = np.array(train_data["c0_anode"])
test_c0_anode = np.array(test_data["c0_anode"])
train_D_anode = np.array(train_data["Dan"])
test_D_anode = np.array(test_data["Dan"])

###Cathode data ###
train_cn_cathode = np.array(train_data["cn_cathode"])
test_cn_cathode = np.array(test_data["cn_cathode"])
train_c0_cathode = np.array(train_data["c0_cathode"])
test_c0_cathode = np.array(test_data["c0_cathode"])
train_D_cathode = np.array(train_data["Dca"])
test_D_cathode = np.array(test_data["Dca"])

In [5]:

params_bat = pybamm.ParameterValues(parameter_name)
cs_max_a = params_bat["Maximum concentration in negative electrode [mol.m-3]"]
cs_max_c = params_bat["Maximum concentration in positive electrode [mol.m-3]"]

cs_max_a_norm = 1.
cs_max_c_norm = 1.
cs_min_a_norm = 0.0
cs_min_c_norm = 0.0

In [6]:
train_cn_anode, train_cn_cathode, train_mask = filter_anode_cathode(train_cn_anode, train_cn_cathode,
                                                               anode_lo=cs_min_a_norm, anode_hi=cs_max_a_norm, 
                                                               cathode_lo=cs_min_c_norm, cathode_hi=cs_max_c_norm)

In [7]:
test_cn_anode, test_cn_cathode, test_mask = filter_anode_cathode(test_cn_anode, test_cn_cathode,
                                                               anode_lo=cs_min_a_norm, anode_hi=cs_max_a_norm, 
                                                               cathode_lo=cs_min_c_norm, cathode_hi=cs_max_c_norm)

In [8]:
train_I = train_I[train_mask]
test_I = test_I[test_mask]
train_c0_anode = train_c0_anode[train_mask]
test_c0_anode = test_c0_anode[test_mask]
train_c0_cathode = train_c0_cathode[train_mask]
test_c0_cathode = test_c0_cathode[test_mask]
train_D_anode = train_D_anode[train_mask]
test_D_anode = test_D_anode[test_mask]
train_D_cathode = train_D_cathode[train_mask]
test_D_cathode = test_D_cathode[test_mask]

In [9]:
train_D_anode = normalise_diffusion(train_D_anode).reshape(-1, 1)
test_D_anode = normalise_diffusion(test_D_anode).reshape(-1, 1)

train_D_cathode = normalise_diffusion(train_D_cathode).reshape(-1, 1)
test_D_cathode = normalise_diffusion(test_D_cathode).reshape(-1, 1)

In [10]:
# Padding amounts
padding_t = 5  # along t-axis
padding_r = 2  # along r-axis

# Original sample counts
num_samples_I = 75
num_samples_c0 = 20

In [11]:
X_test_anode, Y_test_anode = preprocess_data(test_I, test_c0_anode, test_cn_anode, num_samples_I, num_samples_c0, padding_r, padding_t)
X_test_cathode, Y_test_cathode = preprocess_data(test_I, test_c0_cathode, test_cn_cathode, num_samples_I, num_samples_c0, padding_r, padding_t)

In [12]:
# Assume these hyperparameters
k_modes = 10
fno_depth = 8
hidden_channels = 64
input_channels = X_test_anode.shape[-1]  # should be 4
output_channels = 1
cape_hidden_size = 32

In [13]:
model = CAPE_FNO(k_modes=k_modes, input_channels= input_channels, 
                 fno_depth=fno_depth, cape_hidden_size = cape_hidden_size, 
                 hidden_channels=hidden_channels, output_channels=output_channels)

main_key = jax.random.PRNGKey(random_seed)
# Initialize parameters
dummy_D = jax.random.normal(main_key, (1,1))
params = model.init(main_key, X_test_anode[:1,...], dummy_D)

# Forward pass
out = model.apply(params, X_test_anode[:1,...], dummy_D)

In [ ]:
# anode_file = "../trained_models/cape_fno/anode_Chen2020_GRF_33000_2025-06-17_22-31-44.msgpack"
# cathode_file = "../trained_models/cape_fno/cathode_Chen2020_GRF_33000_2025-06-17_23-01-46.msgpack"

# anode_file = "../trained_models/cape_fno/anode_Chen2020_PLS_33000_2025-06-17_21-22-22.msgpack"
# cathode_file = "../trained_models/cape_fno/cathode_Chen2020_PLS_33000_2025-06-17_21-46-34.msgpack"

# anode_file = "../trained_models/cape_fno/anode_Chen2020_Triangle_33000_2025-06-17_23-34-25.msgpack"
# cathode_file = "../trained_models/cape_fno/cathode_Chen2020_Triangle_33000_2025-06-17_23-58-46.msgpack"

# anode_file = "../trained_models/cape_fno/anode_Chen2020_CC_33000_2025-06-18_13-25-53.msgpack"
# cathode_file = "../trained_models/cape_fno/cathode_Chen2020_CC_33000_2025-06-18_13-50-09.msgpack"

anode_file = "../trained_models/cape_fno/anode_Chen2020_CC_33000_2025-06-18_14-11-45.msgpack"


params_anode = functions.load_model_params(anode_file)
params_cathode = functions.load_model_params(cathode_file)

params_anode = flax.serialization.from_bytes(params, params_anode)
params_cathode = flax.serialization.from_bytes(params, params_cathode)

In [15]:
# parameter_name = "Prada2013"
params_bat = pybamm.ParameterValues(parameter_name)

C = params_bat["Nominal cell capacity [A.h]"]
Dan = params_bat["Negative particle diffusivity [m2.s-1]"]
Dca = params_bat["Positive particle diffusivity [m2.s-1]"]
Ran = params_bat["Negative particle radius [m]"]
Rca = params_bat["Positive particle radius [m]"]
epsan = params_bat["Negative electrode active material volume fraction"]
epsca = params_bat["Positive electrode active material volume fraction"]
cs_max_a = params_bat["Maximum concentration in negative electrode [mol.m-3]"]
cs_max_c = params_bat["Maximum concentration in positive electrode [mol.m-3]"]
Lan = params_bat["Negative electrode thickness [m]"]
Lca = params_bat["Positive electrode thickness [m]"]
A = params_bat["Electrode height [m]"] * params_bat["Electrode width [m]"]
t_max = 3600

t = np.linspace(0, 1, num_samples_I)
r = np.linspace(0, 1, num_samples_c0)

In [16]:
X_train_anode, Y_train_anode = preprocess_data(train_I, train_c0_anode, train_cn_anode, num_samples_I, num_samples_c0, padding_r, padding_t)
X_test_anode, Y_test_anode = preprocess_data(test_I, test_c0_anode, test_cn_anode, num_samples_I, num_samples_c0, padding_r, padding_t)

X_train_cathode, Y_train_cathode = preprocess_data(train_I, train_c0_cathode, train_cn_cathode, num_samples_I, num_samples_c0, padding_r, padding_t)
X_test_cathode, Y_test_cathode = preprocess_data(test_I, test_c0_cathode, test_cn_cathode, num_samples_I, num_samples_c0, padding_r, padding_t)

In [17]:
# c_train_true_anode = train_cn_anode
# c_train_pred_anode = model.apply(params_anode,X_train_anode, train_D_anode)
# c_train_true_reshaped_anode = c_train_true_anode  # already (20,75)
# c_train_pred_reshaped_anode = remove_padding(c_train_pred_anode, padding_r, padding_t)

c_test_true_anode = test_cn_anode
c_test_pred_anode = model.apply(params_anode,X_test_anode, test_D_anode)
c_test_true_reshaped_anode = c_test_true_anode   # already (20,75)
c_test_pred_reshaped_anode = remove_padding(c_test_pred_anode, padding_r, padding_t)

In [18]:
diff = c_test_true_anode - c_test_pred_reshaped_anode.squeeze()

In [19]:
# c_train_true_cathode = train_cn_cathode
# c_train_pred_cathode = model.apply(params_cathode,X_train_cathode, train_D_cathode)
# c_train_true_reshaped_cathode = c_train_true_cathode  # already (20,75)
# c_train_pred_reshaped_cathode = remove_padding(c_train_pred_cathode, padding_r, padding_t)

c_test_true_cathode = test_cn_cathode
c_test_pred_cathode = model.apply(params_cathode,X_test_cathode, test_D_cathode)
c_test_true_reshaped_cathode = c_test_true_cathode   # already (20,75)
c_test_pred_reshaped_cathode = remove_padding(c_test_pred_cathode, padding_r, padding_t)

In [20]:
# diff = c_test_true_anode - c_test_pred_reshaped_anode.squeeze()
# diff

In [21]:
#c_train_pred_scaled_anode = c_train_pred_reshaped_anode * cs_max_a
#c_train_true_scaled_anode = c_train_true_reshaped_anode * cs_max_a
#c_train_pred_scaled_cathode = c_train_pred_reshaped_cathode * cs_max_c
#c_train_true_scaled_cathode = c_train_true_reshaped_cathode * cs_max_c

c_test_pred_scaled_anode = c_test_pred_reshaped_anode * cs_max_a
c_test_true_scaled_anode = c_test_true_reshaped_anode * cs_max_a
c_test_pred_scaled_cathode = c_test_pred_reshaped_cathode * cs_max_c
c_test_true_scaled_cathode = c_test_true_reshaped_cathode * cs_max_c

In [22]:
c_pred_an_surf = c_test_pred_reshaped_anode[:,-1,:].squeeze()
c_true_an_surf = c_test_true_reshaped_anode[:,-1,:].squeeze()
c_pred_ca_surf = c_test_pred_reshaped_cathode[:,-1,:].squeeze()
c_true_ca_surf = c_test_true_reshaped_cathode[:,-1,:].squeeze()

In [23]:
# Compute V_pred and V_true using post_proc function
#from functions import post_proc
V_pred, V_true = functions.post_proc(params_bat, test_I, c_pred_an_surf, c_true_an_surf, c_pred_ca_surf, c_true_ca_surf, Ran, Rca, epsan, epsca, Lan, Lca, A)

In [24]:
V_max = params_bat["Upper voltage cut-off [V]"]
V_min = params_bat["Lower voltage cut-off [V]"]
V_pred_norm = (V_pred - V_min) / (V_max - V_min)
V_true_norm = (V_true - V_min) / (V_max - V_min)

In [25]:
concentration_errors_anode = calc_error_metrics(c_test_pred_scaled_anode, c_test_true_scaled_anode)
concentration_errors_cathode = calc_error_metrics(c_test_pred_scaled_cathode, c_test_true_scaled_cathode)
concentration_errors_all = calc_error_metrics_all(concentration_errors_anode, concentration_errors_cathode)
voltage_errors = calc_error_metrics(V_pred, V_true, axis=(1,))

In [26]:
concentration_errors_anode_norm = calc_error_metrics(c_test_pred_reshaped_anode, c_test_true_reshaped_anode)
concentration_errors_cathode_norm = calc_error_metrics(c_test_pred_reshaped_cathode, c_test_true_reshaped_cathode)
concentration_errors_all_norm = calc_error_metrics_all(concentration_errors_anode_norm, concentration_errors_cathode_norm)
voltage_errors_norm = calc_error_metrics(V_pred_norm, V_true_norm, axis=(1,))

In [27]:
voltage_errors["mse"]

Array([1.82662848e-06, 1.80424547e-06, 1.78469869e-04, 3.47458590e-05,
       2.26757576e-04, 4.17525007e-05, 3.76407183e-06, 1.23170859e-04,
       2.51770985e-06, 1.12713155e-06, 3.70591442e-04, 1.00675585e-04,
       9.00277591e-05, 1.37310475e-04, 3.74986936e-04, 2.76500487e-06,
       5.75795857e-05, 2.06295037e-04, 2.05649030e-05, 1.17557038e-05,
       1.60758005e-04, 1.15720095e-05, 4.20985407e-06, 1.24953585e-04,
       2.43620107e-05, 3.61027992e-06, 1.07461146e-05, 4.74608350e-06,
       6.16784246e-06, 1.30225669e-06, 9.29648104e-06, 1.06372056e-04,
       4.47666716e-05, 5.74407750e-05, 8.18792105e-05, 1.97501231e-06,
       7.72348067e-05, 6.68355933e-05, 2.40201712e-06, 4.91204555e-05,
       7.20462049e-05, 1.95676262e-06, 4.60576211e-06, 8.05820528e-05,
       1.61009064e-06, 5.89922383e-05, 1.37781435e-05, 8.56140250e-05,
       7.85691282e-06, 2.06674417e-06, 5.90761701e-05, 1.53366345e-05,
       4.57624992e-06, 5.23667950e-06, 8.38815322e-06, 1.22571043e-06,
      

In [28]:
concentration_errors_anode_norm["rel_l2"].mean() * 100

Array(0.22597271, dtype=float32)

In [29]:
a = concentration_errors_all["mse"]

In [30]:
np.sqrt(a)

array([ 62.90222 ,  63.404903, 170.57707 , 118.30069 , 192.09988 ,
        70.221535,  52.261578,  59.678562,  58.622562,  55.834618,
        56.749084, 106.47249 , 101.48503 ,  63.839214,  67.52893 ,
        47.112057, 105.29058 ,  78.423965, 111.947525,  62.769268,
       143.11261 ,  79.05375 ,  61.275093, 116.92626 ,  53.57296 ,
        69.927246,  74.56456 ,  56.327763,  82.51895 ,  54.229225,
        61.30056 , 133.05653 ,  87.32605 ,  68.875565,  74.25841 ,
        41.87482 , 136.43579 ,  88.94542 ,  55.522182, 104.40647 ,
       132.69406 ,  56.618362,  64.79571 ,  58.200615,  66.570335,
        75.537155,  68.19491 , 174.66249 ,  96.098976,  54.824818,
        64.97444 ,  58.847523,  70.60497 ,  73.20445 ,  69.99465 ,
        61.62286 ,  99.54524 ,  52.52012 ,  83.22252 ,  52.28853 ,
        60.722763,  62.262684,  79.1769  ,  74.42684 , 145.7393  ,
        55.66234 , 111.59111 ,  55.496517,  55.641884,  57.219448,
       105.30539 ,  58.970848, 101.386375,  64.333115,  71.069

In [31]:
concentration_errors_all_norm["mse"].mean()

Array(2.3863593e-06, dtype=float32)

In [32]:
voltage_errors["mae"].mean() * 1e3

Array(4.3980956, dtype=float32)

In [33]:
for  key, value in concentration_errors_all.items():

    if key == "mse":
        value = np.sqrt(value)
    if key == "rel_l2":
        value = value * 100
    if key == "mae":
        value = value
    if key == "rel_linf":
        value = value * 100

    print(f"{key}: {value.mean():.4f}")

mae: 55.5829
mse: 81.1835
rel_l2: 0.2433
rel_linf: 0.4181


In [34]:
for  key, value in voltage_errors.items():

    if key == "mse":
        value = np.sqrt(value)*1000
    if key == "rel_l2":
        value = value * 100
    if key == "mae":
        value = value * 1e3
    if key == "rel_linf":
        value = value * 100

    print(f"{key}: {value.mean():.4f}")

mae: 4.3981
mse: 6.9017
rel_l2: 0.1937
rel_linf: 0.7705
